<a href="https://colab.research.google.com/github/DhananjayaFdo/Databases-and-Analytics-Assignment/blob/main/sql_in_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Install and load the package
install.packages("sqldf")
library(sqldf)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [10]:
print("Loading files...")

baseUrl = "https://raw.githubusercontent.com/DhananjayaFdo/Databases-and-Analytics-Assignment/refs/heads/main/northstar_dataset";

hubs <- read.csv(paste0(baseUrl, '/hubs.csv'))
customers <- read.csv(paste0(baseUrl, '/customers.csv'))
drivers <- read.csv(paste0(baseUrl, '/drivers.csv'));
vehicles <- read.csv (paste0(baseUrl, '/vehicles.csv'));
orders  <- read.csv (paste0(baseUrl, '/orders.csv'));
deliveries  <- read.csv (paste0(baseUrl, '/deliveries.csv'));
incidents  <- read.csv (paste0(baseUrl, '/incidents.csv'));
complaints  <- read.csv (paste0(baseUrl, '/complaints.csv'));
app_events <- read.csv (paste0(baseUrl, '/app_events.csv'));

print("All files loaded ✓")

[1] "Loading files..."
[1] "All files loaded ✓"


# How many deliveries failed, were delayed, or on time?

In [14]:
q1 <- sqldf("
  SELECT delivery_status,
         COUNT(*) AS total
  FROM deliveries
  GROUP BY delivery_status
  ORDER BY total DESC
")

print(q1)

  delivery_status total
1          OnTime   616
2         Delayed   202
3          Failed   132


"132 deliveries failed and 202 were delayed — together that's 35% of all deliveries. This directly supports the operations director's concern about service reliability."

# Which service type has the most failed deliveries?

In [15]:
q2 <- sqldf("
  SELECT o.service_type,
         COUNT(*) AS failed_count
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  WHERE d.delivery_status = 'Failed'
  GROUP BY o.service_type
  ORDER BY failed_count DESC
")

print(q2)

  service_type failed_count
1    Passenger           38
2       Retail           28
3       Parcel           25
4     Business           25
5      Medical           16


"This reveals which service lines are driving failures — helping the finance director understand where losses are really coming from."

# Which zone has the worst delivery performance?

In [16]:
q3 <- sqldf("
  SELECT o.pickup_zone,
         COUNT(*) AS total_orders,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY failed DESC
")

print(q3)

   pickup_zone total_orders failed avg_rating
1    RiverSide           66     14       3.80
2         EAST           78     11       3.86
3          Ctr           64     11       3.43
4      Central           55     11       3.59
5      CENTRAL           55     11       3.64
6        South           83     10       3.99
7        north           52      8       3.94
8         East           78      8       3.96
9      Airport           67      8       3.86
10        West           51      7       3.94
11        WEST           63      7       3.86
12       North           37      7       3.84
13       NORTH           46      7       3.89
14       SOUTH           56      4       4.14
15   Riverside           53      4       3.95
16     AIRPORT           46      4       4.16


"Zones with high failure counts and low ratings flag exactly what the operations director suspects — certain city areas are consistently underperforming."

# Do experienced drivers have fewer failed deliveries?

In [17]:
q4 <- sqldf("
  SELECT
    CASE
      WHEN dr.years_experience < 3 THEN 'Junior (0-2 yrs)'
      WHEN dr.years_experience < 7 THEN 'Mid (3-6 yrs)'
      ELSE 'Senior (7+ yrs)'
    END AS experience_group,
    COUNT(*) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
    ROUND(AVG(dr.driver_rating), 2) AS avg_driver_rating
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY experience_group
  ORDER BY failed DESC
")

print(q4)

  experience_group total_deliveries failed avg_driver_rating
1  Senior (7+ yrs)              614     97              4.19
2    Mid (3-6 yrs)              232     26              4.03
3 Junior (0-2 yrs)              104      9              4.31


"This tests whether driver experience actually affects service quality, which is relevant to NorthStar's staffing and training decisions."

# Which hubs generate the most complaints?

In [18]:
q5 <- sqldf("
  SELECT h.hub_name,
         h.zone,
         COUNT(c.complaint_id) AS total_complaints,
         ROUND(AVG(c.resolution_days), 1) AS avg_resolution_days
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  JOIN orders o ON d.order_id = o.order_id
  JOIN complaints c ON c.order_id = o.order_id
  GROUP BY h.hub_name, h.zone
  ORDER BY total_complaints DESC
")

print(q5)

        hub_name      zone total_complaints avg_resolution_days
1  Midtown Relay   Central               35                 8.1
2      East Dock      East               33                 7.9
3  Riverside Hub Riverside               33                 6.5
4 North Exchange     North               32                 7.9
5   Central Core   Central               30                 9.4
6      West Gate      West               28                 7.2
7    Airport Hub   Airport               23                 9.1
8     South Link     South               18                 7.5


"Hubs with high complaint volumes and slow resolution are the hidden problem areas the customer experience director is concerned about."

# What is the average order value by priority level?

In [19]:
q6 <- sqldf("
  SELECT priority_level,
         COUNT(*) AS total_orders,
         ROUND(AVG(order_value), 2) AS avg_value,
         ROUND(SUM(order_value), 2) AS total_value
  FROM orders
  GROUP BY priority_level
  ORDER BY avg_value DESC
")

print(q6)

  priority_level total_orders avg_value total_value
1           High          308     95.66    29464.42
2         Medium          503     90.18    45361.87
3       Critical           91     89.38     8133.99
4            Low          348     88.66    30852.87


"Critical priority orders carry the highest value — if these are also the ones being delayed or failed, the financial impact on NorthStar is significant."